# Revision Analyses — Colab GPU Version

**Manuscript:** BIOINF-2026-0189 (Bioinformatics major revision)

This notebook runs all revision analyses on GPU:
1. **W5** — Subtype-stratified performance + temporal holdout
2. **W6** — Multi-PLM comparison (ESM-2 vs ESM C 600M vs ESM-1v)

**Runtime:** ~15-20 minutes on Colab T4 GPU

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q esm torch transformers xgboost scikit-learn seaborn openpyxl biopython tqdm

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Clone repo and upload data ──────────────────────────────────────────
!git clone https://github.com/hayden-farquhar/HIV-ESM-2.git

import sys, os
os.chdir('HIV-ESM-2')
sys.path.insert(0, '.')

# Create directories
os.makedirs('data/embeddings', exist_ok=True)
os.makedirs('results/revision', exist_ok=True)
os.makedirs('figures/revision', exist_ok=True)

In [ ]:
# ── Upload the 3 HIVDB data files ──────────────────────────────────────
# Upload PI_dataset.txt, NRTI_dataset.txt, NNRTI_dataset.txt
from google.colab import files
print('Upload PI_dataset.txt, NRTI_dataset.txt, NNRTI_dataset.txt')
uploaded = files.upload()

# Move to a data_raw directory
os.makedirs('data_raw', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'data_raw/{fname}')
    print(f'  Moved {fname} -> data_raw/{fname}')

from pathlib import Path
DATA_DIR = Path('data_raw')

---
## Part 1: Subtype Analysis + Temporal Holdout (W5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from src.data_processing import PI_DRUGS, NRTI_DRUGS, NNRTI_DRUGS
from src.subtype_analysis import (
    reconstruct_all_datasets,
    assign_subtypes_via_sequence_similarity,
    subtype_stratified_evaluation,
    create_temporal_split,
    temporal_holdout_evaluation,
)

RESULTS_DIR = Path('results/revision')
FIGURES_DIR = Path('figures/revision')
EMBEDDING_DIR = Path('data/embeddings')

# Step 1: Reconstruct sequences
print('=== Reconstructing sequences ===')
datasets = reconstruct_all_datasets(DATA_DIR)
for dc, df in datasets.items():
    print(f'  {dc}: {len(df)} sequences, length={df["sequence"].str.len().iloc[0]}')

In [ ]:
# Step 2: Assign subtypes
print('=== Assigning subtypes ===')
subtype_dfs = {}
for dc, df in datasets.items():
    st_df = assign_subtypes_via_sequence_similarity(
        df['sequence'].tolist(), df['SeqID'].tolist(), drug_class=dc
    )
    subtype_dfs[dc] = st_df
    print(f'\n{dc}:')
    print(st_df['subtype'].value_counts())

# Save
all_st = pd.concat([s.assign(drug_class=dc) for dc, s in subtype_dfs.items()])
all_st.to_csv(RESULTS_DIR / 'subtype_assignments.csv', index=False)

In [ ]:
# Step 3: Extract ESM-2 embeddings (GPU — fast!)
from transformers import EsmModel, EsmTokenizer

print('Loading ESM-2...')
tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
esm2_model = EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D').to(device).eval()
print(f'ESM-2 loaded on {device}')

def extract_embeddings_hf(sequences, model, tokenizer, device, batch_size=16):
    """Extract mean-pooled embeddings via HuggingFace transformers."""
    all_pooled = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=1024)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for j, seq in enumerate(batch):
            emb = out.last_hidden_state[j, 1:len(seq)+1, :].cpu().numpy()
            all_pooled.append(emb.mean(axis=0))
        if device.type == 'cuda': torch.cuda.empty_cache()
    return np.array(all_pooled)

esm2_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esm2_embeddings.npy'
    if cache.exists():
        esm2_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esm2_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting...')
        X = extract_embeddings_hf(df['sequence'].tolist(), esm2_model, tokenizer, device, batch_size=16)
        np.save(cache, X)
        esm2_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

del esm2_model
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# Step 4: Binarize phenotypes
drug_lists = {'PI': PI_DRUGS, 'NRTI': NRTI_DRUGS, 'NNRTI': NNRTI_DRUGS}
pheno_dfs = {}
for dc, df in datasets.items():
    pheno = pd.DataFrame()
    for drug in drug_lists[dc]:
        if drug in df.columns:
            fc = pd.to_numeric(df[drug], errors='coerce')
            pheno[drug] = np.where(fc.isna(), np.nan, (fc >= 2.5).astype(float))
    pheno_dfs[dc] = pheno
    print(f'{dc}: {len(pheno)} samples')

In [ ]:
# Step 5: Subtype-stratified evaluation
print('=== Subtype-stratified AUC ===')
all_strat = []
for dc in datasets:
    print(f'\n--- {dc} ---')
    strat = subtype_stratified_evaluation(
        esm2_embeddings[dc], pheno_dfs[dc], subtype_dfs[dc]['subtype'],
        drug_lists[dc], model_type='logistic'
    )
    strat['drug_class'] = dc
    all_strat.append(strat)

strat_df = pd.concat(all_strat, ignore_index=True)
strat_df.to_csv(RESULTS_DIR / 'subtype_stratified_auc.csv', index=False)
print('\nMean AUC by subtype:')
print(strat_df.groupby('subtype')['auc'].agg(['mean', 'std', 'count']))

In [ ]:
# Step 6: Temporal holdout
print('=== Temporal holdout ===')
all_temp = []
for dc in datasets:
    print(f'\n--- {dc} ---')
    train_idx, test_idx = create_temporal_split(datasets[dc], cutoff_quantile=0.8)
    temp = temporal_holdout_evaluation(
        esm2_embeddings[dc], pheno_dfs[dc], drug_lists[dc],
        train_idx, test_idx, model_type='logistic'
    )
    temp['drug_class'] = dc
    all_temp.append(temp)

temp_df = pd.concat(all_temp, ignore_index=True)
temp_df.to_csv(RESULTS_DIR / 'temporal_holdout_auc.csv', index=False)
print(f'\nMean temporal holdout AUC: {temp_df["auc"].mean():.4f}')
print(temp_df[['drug_class', 'drug', 'auc', 'n_train', 'n_test']].to_string(index=False))

---
## Part 2: Multi-PLM Comparison (W6)

In [ ]:
# ── ESM C 600M embeddings ───────────────────────────────────────────────
from esm.models.esmc import ESMC
from esm.tokenization import EsmSequenceTokenizer

print('Loading ESM C 600M...')
esmc_model = ESMC.from_pretrained('esmc_600m', device=device)
esmc_model.eval()
esmc_tokenizer = EsmSequenceTokenizer()
print(f'ESM C loaded on {device}')

def extract_esmc_embs(sequences, model, tokenizer, device, batch_size=16):
    all_pooled = []
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch = sequences[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, return_tensors='pt')
        input_ids = tokens['input_ids'].to(device)
        with torch.no_grad():
            out = model(input_ids)
        for j, seq in enumerate(batch):
            emb = out.embeddings[j, 1:len(seq)+1, :].cpu().float().numpy()
            all_pooled.append(emb.mean(axis=0))
        if device.type == 'cuda': torch.cuda.empty_cache()
    return np.array(all_pooled)

esmc_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esmc_embeddings.npy'
    if cache.exists():
        esmc_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esmc_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting ESM C embeddings...')
        X = extract_esmc_embs(df['sequence'].tolist(), esmc_model, esmc_tokenizer, device)
        np.save(cache, X)
        esmc_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

del esmc_model
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ── ESM-1v embeddings ───────────────────────────────────────────────────
print('Loading ESM-1v...')
esm1v_tokenizer = EsmTokenizer.from_pretrained('facebook/esm1v_t33_650M_UR90S_1')
esm1v_model = EsmModel.from_pretrained('facebook/esm1v_t33_650M_UR90S_1').to(device).eval()
print(f'ESM-1v loaded on {device}')

esm1v_embeddings = {}
for dc, df in datasets.items():
    cache = EMBEDDING_DIR / f'{dc}_esm1v_embeddings.npy'
    if cache.exists():
        esm1v_embeddings[dc] = np.load(cache)
        print(f'{dc}: loaded cached {esm1v_embeddings[dc].shape}')
    else:
        print(f'{dc}: extracting ESM-1v embeddings...')
        X = extract_embeddings_hf(df['sequence'].tolist(), esm1v_model, esm1v_tokenizer, device, batch_size=16)
        np.save(cache, X)
        esm1v_embeddings[dc] = X
        print(f'  {dc}: {X.shape}')

del esm1v_model
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ── Run classifier comparison across all PLMs ───────────────────────────
from src.models import per_drug_training

all_plm_results = []

for plm_name, emb_dict in [('ESM-2', esm2_embeddings), ('ESM-C', esmc_embeddings), ('ESM-1v', esm1v_embeddings)]:
    print(f'\n=== {plm_name} ===')
    for dc in ['PI', 'NRTI', 'NNRTI']:
        X = emb_dict[dc]
        drugs = drug_lists[dc]
        pheno = pheno_dfs[dc]
        print(f'\n  {dc} ({X.shape[1]}-dim):')
        results = per_drug_training(X, pheno, drugs, model_type='logistic', n_splits=5, random_state=42)
        for drug, res in results.items():
            all_plm_results.append({
                'plm': plm_name, 'drug_class': dc, 'drug': drug,
                'auc': res['auc'], 'n_samples': res['n_samples'],
                'embed_dim': X.shape[1]
            })

plm_df = pd.DataFrame(all_plm_results)
plm_df.to_csv(RESULTS_DIR / 'plm_comparison_results.csv', index=False)

print('\n=== Mean AUC by PLM ===')
print(plm_df.groupby('plm')['auc'].agg(['mean', 'std']).round(4))

In [ ]:
# ── Comparison table ────────────────────────────────────────────────────
pivot = plm_df.pivot_table(index='drug', columns='plm', values='auc')
mean_row = pivot.mean().to_frame('MEAN').T
pivot = pd.concat([pivot, mean_row])
print('\n=== PLM Comparison Table ===')
print(pivot.round(4).to_string())
pivot.to_csv(RESULTS_DIR / 'plm_comparison_table.csv')

In [ ]:
# ── Figures ─────────────────────────────────────────────────────────────

# Fig: PLM comparison bar + box
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart by drug
ax = axes[0]
piv = plm_df.pivot(index='drug', columns='plm', values='auc').sort_index()
piv.plot(kind='bar', ax=ax, width=0.8, edgecolor='white')
ax.set_ylabel('AUC-ROC', fontsize=13)
ax.set_title('PLM Comparison: AUC by Drug', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])
ax.legend(title='PLM')
ax.tick_params(axis='x', rotation=45)

# Box plot
ax = axes[1]
sns.boxplot(data=plm_df, x='plm', y='auc', ax=ax, palette='Set2')
sns.stripplot(data=plm_df, x='plm', y='auc', ax=ax, color='black', alpha=0.4, size=4)
ax.set_ylabel('AUC-ROC', fontsize=13)
ax.set_title('PLM Comparison: Distribution Across 18 Drugs', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])
for i, plm in enumerate(plm_df['plm'].unique()):
    m = plm_df[plm_df['plm']==plm]['auc'].mean()
    ax.text(i, 0.81, f'mean={m:.3f}', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'plm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 10))
drug_order = PI_DRUGS + NRTI_DRUGS + NNRTI_DRUGS
piv_hm = plm_df.pivot(index='drug', columns='plm', values='auc')
piv_hm = piv_hm.reindex([d for d in drug_order if d in piv_hm.index])
sns.heatmap(piv_hm, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.85, vmax=1.0, ax=ax, linewidths=0.5)
ax.set_title('Multi-PLM Performance Comparison', fontsize=14, fontweight='bold')
n_pi = len([d for d in PI_DRUGS if d in piv_hm.index])
n_nrti = len([d for d in NRTI_DRUGS if d in piv_hm.index])
ax.axhline(y=n_pi, color='black', linewidth=2)
ax.axhline(y=n_pi+n_nrti, color='black', linewidth=2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'plm_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Subtype + temporal figures ──────────────────────────────────────────

# Subtype-stratified AUC
if not strat_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_data = strat_df.dropna(subset=['auc'])
    sns.boxplot(data=plot_data, x='subtype', y='auc', hue='drug_class', ax=ax, palette='Set2')
    ax.set_ylabel('AUC-ROC', fontsize=13)
    ax.set_title('ESM-2 Performance by HIV Subtype', fontsize=14, fontweight='bold')
    ax.set_ylim([0.7, 1.0])
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'subtype_stratified_performance.png', dpi=300, bbox_inches='tight')
    plt.show()

# Temporal holdout
if not temp_df.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    colors_map = {'PI': '#2196F3', 'NRTI': '#4CAF50', 'NNRTI': '#FF9800'}
    bar_colors = [colors_map.get(r['drug_class'], '#999') for _, r in temp_df.iterrows()]
    bars = ax.bar(range(len(temp_df)), temp_df['auc'], color=bar_colors, edgecolor='white')
    ax.set_xticks(range(len(temp_df)))
    ax.set_xticklabels(temp_df['drug'], rotation=45, ha='right')
    ax.set_ylabel('AUC', fontsize=13)
    ax.set_title('Temporal Holdout AUC', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.05)
    for bar, val in zip(bars, temp_df['auc']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'temporal_holdout_auc.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Save everything to Excel + download ─────────────────────────────────
with pd.ExcelWriter(RESULTS_DIR / 'revision_results.xlsx') as w:
    strat_df.to_excel(w, sheet_name='Subtype_Stratified', index=False)
    temp_df.to_excel(w, sheet_name='Temporal_Holdout', index=False)
    plm_df.to_excel(w, sheet_name='PLM_Comparison', index=False)
    pivot.to_excel(w, sheet_name='PLM_Table')
    all_st.to_excel(w, sheet_name='Subtype_Assignments', index=False)

print('Results saved to results/revision/revision_results.xlsx')

# Print summary
print('\n' + '='*60)
print('SUMMARY')
print('='*60)

print('\n--- Subtype-stratified AUC ---')
print(strat_df.groupby('subtype')['auc'].agg(['mean', 'std', 'count']).round(4))

print(f'\n--- Temporal holdout: mean AUC = {temp_df["auc"].mean():.4f} ---')

print('\n--- PLM comparison ---')
print(plm_df.groupby('plm')['auc'].agg(['mean', 'std']).round(4))

print('\n--- Drugs where ESM C > ESM-2 ---')
e2 = plm_df[plm_df['plm']=='ESM-2'].set_index('drug')['auc']
ec = plm_df[plm_df['plm']=='ESM-C'].set_index('drug')['auc']
for d in e2.index.intersection(ec.index):
    diff = ec[d] - e2[d]
    print(f'  {d}: {e2[d]:.4f} -> {ec[d]:.4f} ({diff:+.4f})')

In [ ]:
# Download results
from google.colab import files
files.download('results/revision/revision_results.xlsx')